# Lab 12 — 세 가지 방법으로 같은 MLE 구하기

**확률통계 · Topic 12 · 부산대학교 정보컴퓨터공학부**

---

### 오늘의 목표

1. 같은 MLE를 **격자 탐색 / 해석적 해 / 수치 최적화** 세 가지로 구해 일치를 확인한다.
2. **log-likelihood를 쓰는 이유**(언더플로)를 직접 겪는다.
3. 표본분산의 **$n$ vs $n-1$ 편향**을 시뮬레이션으로 확인한다.

⏱ **예상 소요 시간: 35분**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize

rng = np.random.default_rng(20260302)

# 오늘의 데이터: 동전 100번, 앞면 63번
n_trials, n_heads = 100, 63
print(f"관측: {n_trials}번 중 앞면 {n_heads}번")

## 방법 1. 격자 탐색 — 훑어서 찾기

$\theta$ 를 잘게 나눠 각각의 likelihood를 계산하고 **가장 큰 것**을 고른다.
원시적이지만 **언제나 통한다.**

### 실습 1

In [ ]:
theta = np.linspace(0.001, 0.999, 2000)
lik = stats.binom.pmf(n_heads, n_trials, theta)

# TODO 1: likelihood가 가장 큰 theta를 고르세요.  힌트: theta[np.argmax(lik)]
theta_grid = 0.5

plt.figure(figsize=(7, 4))
plt.plot(theta, lik)
plt.axvline(theta_grid, color="red", ls="--")
plt.xlabel("theta")
plt.ylabel("likelihood")
plt.title(f"Grid search: theta_hat = {theta_grid:.4f}")
plt.show()

print(f"격자 탐색 결과: {theta_grid:.6f}")

## 방법 2. 해석적 해 — 미분해서 푼 공식

$$\ell(\theta) = k\log\theta + (n-k)\log(1-\theta)
\;\Rightarrow\; \hat\theta = \frac{k}{n}$$

### 실습 2

In [ ]:
# TODO 2: 해석적 해 k/n 을 계산하세요
theta_analytic = 0.0

print(f"해석적 해: {theta_analytic:.6f}")
print(f"격자와의 차이: {abs(theta_analytic - theta_grid):.6f}")

## 방법 3. 수치 최적화 — 컴퓨터에 맡기기

**음의** log-likelihood를 최소화한다 (최적화 라이브러리는 최소화만 한다).

### 실습 3

In [ ]:
def neg_loglik(theta):
    if theta <= 0 or theta >= 1:
        return np.inf
    # TODO 3: 음의 log-likelihood 를 돌려주세요
    #         힌트: -(k*log(theta) + (n-k)*log(1-theta))
    return 0.0


res = optimize.minimize_scalar(neg_loglik, bounds=(0.001, 0.999), method="bounded")
theta_numeric = res.x

print(f"수치 최적화 결과: {theta_numeric:.6f}")
print(f"\n세 방법 비교")
print(f"  격자     {theta_grid:.6f}")
print(f"  해석적   {theta_analytic:.6f}")
print(f"  수치     {theta_numeric:.6f}")

> 세 값이 소수점 아래까지 일치하는가?
> **격자는 촘촘함에 따라, 수치 최적화는 수렴 기준에 따라** 미세하게 다를 수 있다.

## Part 2. 왜 log를 쓰는가 — 직접 겪어보기

표본을 늘려가며 likelihood를 **곱으로** 계산해보자.

### 실습 4

In [ ]:
print(f"{'n':>7}{'likelihood (곱)':>20}{'log-likelihood':>18}")
for n in [50, 200, 800, 5000, 20000]:
    x = (rng.random(n) < 0.63).astype(int)
    k = x.sum()
    p = k / n
    lik = p ** k * (1 - p) ** (n - k)
    # TODO 4: log-likelihood 를 계산하세요
    #         힌트: k*np.log(p) + (n-k)*np.log(1-p)
    loglik = 0.0
    print(f"{n:>7}{lik:>20.3e}{loglik:>18.2f}")

😱 **$n$ 이 커지면 likelihood가 정확히 0이 된다.**

컴퓨터의 실수 표현 한계(약 $10^{-308}$) 아래로 내려가 **언더플로**가 일어난 것이다.
0이 되어버리면 최대점을 찾을 수 없다.

**log-likelihood는 −2만 정도의 값으로 멀쩡히 살아 있다.**
이것이 실무에서 항상 log를 쓰는 이유다.

## Part 3. 표본분산의 편향

Gaussian의 MLE는 $\hat\sigma^2 = \frac{1}{n}\sum(x_i - \bar{x})^2$ 였다.
**$n$ 으로 나눈다.** 그런데 우리는 보통 $n-1$ 로 나눈다고 배웠다. 무엇이 맞을까?

### 실습 5 — 어느 쪽이 참값에 가까운가

In [ ]:
TRUE_VAR = 4.0
REPEATS = 20000

print(f"참 분산 = {TRUE_VAR}\n")
print(f"{'n':>5}{'MLE (ddof=0)':>16}{'불편 (ddof=1)':>16}")

for n in [2, 3, 5, 10, 30, 100]:
    x = rng.normal(0, np.sqrt(TRUE_VAR), size=(REPEATS, n))
    # TODO 5: ddof=0 과 ddof=1 로 각각 분산을 구해 평균 내세요
    #         힌트: x.var(axis=1, ddof=0).mean()
    v_mle = 0.0
    v_unb = 0.0
    print(f"{n:>5}{v_mle:>16.4f}{v_unb:>16.4f}")

🤔 **MLE는 분산을 계속 작게 추정한다.** $n$ 이 작을수록 심하다.

이유: 표본평균 $\bar{x}$ 는 **그 표본에 가장 잘 맞는 중심**이다.
참 평균 $\mu$ 대신 $\bar{x}$ 를 쓰면 편차가 실제보다 작아진다.

보정하면 $\mathbb{E}[\hat\sigma^2_{\text{MLE}}] = \frac{n-1}{n}\sigma^2$ 이므로,
$n-1$ 로 나누면 정확히 불편이 된다.

> 📌 **`np.var(x)` 는 ddof=0 (MLE), `np.var(x, ddof=1)` 이 불편추정량이다.**
> pandas의 `.var()` 는 기본이 ddof=1 이다. **라이브러리마다 다르니 확인하는 습관을 들일 것.**

### 실습 보너스 — MLE가 무너지는 순간

In [ ]:
# 동전을 3번 던져 3번 다 앞면이 나왔다면?
for k, n in [(3, 3), (0, 3), (5, 5)]:
    print(f"{n}번 중 {k}번 앞면 -> MLE = {k / n}")

print("\n'절대 뒤면이 안 나온다'는 결론이 합리적인가?")
print("다음 주 베이지안 추론이 이 문제를 다룬다.")

---

## 마무리 — 자가 점검

- [ ] 격자 / 해석 / 수치 세 방법으로 같은 MLE를 구했다
- [ ] likelihood가 언더플로로 0이 되는 것을 직접 보았다
- [ ] 표본분산의 $n$ vs $n-1$ 차이를 시뮬레이션으로 확인했다
- [ ] MLE가 작은 표본에서 극단적인 답을 줄 수 있음을 안다

**"딥러닝 학습 = likelihood 최대화" 라는 문장을 자기 말로 풀어써보자.**

> (여기에 작성)